In [ ]:
!pip install yfinance plotly ipywidgets -q

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

import yfinance as yf
import time
from datetime import datetime, timedelta, timezone
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


class TradingBot:

    def __init__(self, tickers, short_window, long_window, sma_windows=None):
        self.tickers = tickers
        self.short_window = short_window
        self.long_window = long_window
        base = sma_windows or [short_window, long_window]
        self.sma_windows = sorted(set(base + [short_window, long_window]))
        self.positions = {ticker: 0 for ticker in self.tickers}
        self._cache = {}

    def get_data(self, ticker):
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            hist = stock.history(period="12mo", interval="1d")
            if hist.empty:
                return None
            return info, hist
        except Exception as e:
            print(f"[ERROR] {ticker}: {e}")
            return None

    def analyze_market(self, ticker, df):
        df['SMA_Short'] = df['Close'].rolling(window=self.short_window).mean()
        df['SMA_Long']  = df['Close'].rolling(window=self.long_window).mean()
        if len(df) < self.long_window:
            return 'HOLD'
        prev_short = df['SMA_Short'].iloc[-2]
        curr_short = df['SMA_Short'].iloc[-1]
        prev_long  = df['SMA_Long'].iloc[-2]
        curr_long  = df['SMA_Long'].iloc[-1]
        if prev_short <= prev_long and curr_short > curr_long:
            return 'BUY'
        elif prev_short >= prev_long and curr_short < curr_long:
            return 'SELL'
        return 'HOLD'

    def get_company_news(self, ticker):
        try:
            stock = yf.Ticker(ticker)
            news_list = stock.news
            print(f"\n--- Latest News for {ticker} ---")
            if news_list:
                for item in news_list[:2]:
                    print(f"- {item.get('title', 'No Title')}")
                    print(f"  Link: {item.get('link', 'No Link')}")
            else:
                print("No recent news found.")
            print("----------------------------------\n")
        except Exception as e:
            print(f"Could not fetch news for {ticker}: {e}")

    def execute_trade(self, ticker, company_name, action, price):
        print(f"ticker       : {ticker}")
        print(f"company_name : {company_name}")
        print(f"action       : {action}")
        print(f"price        : \u20ac{price}")

    def run(self):
        try:
            for ticker in self.tickers:
                data = self.get_data(ticker)
                if data is None:
                    continue
                (info, df) = data
                current_price = df['Close'].iloc[-1]
                signal = self.analyze_market(ticker, df)
                company_name = info.get('longName', ticker)
                if signal in ['BUY', 'SELL']:
                    self.execute_trade(ticker, company_name, signal, current_price)
                else:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] {ticker} [{company_name}]: "
                          f"\u20ac{current_price:.2f} | Strategy: HOLD")
        except KeyboardInterrupt:
            print("\nBot stopped by user.")

    def _get_computed_df(self, ticker):
        if ticker in self._cache:
            return self._cache[ticker]
        data = self.get_data(ticker)
        if data is None:
            return None
        info, df = data
        if df.empty:
            return None
        df = df.copy()
        for w in self.sma_windows:
            df[f'SMA_{w}'] = df['Close'].rolling(window=w).mean()
        df['Signal'] = 0.0
        df.loc[df[f'SMA_{self.short_window}'] > df[f'SMA_{self.long_window}'], 'Signal'] = 1.0
        df['Position'] = df['Signal'].diff()
        company_name = info.get('longName', ticker)
        self._cache[ticker] = (company_name, df)
        return self._cache[ticker]

    def _get_historical_earnings_dates(self, ticker: str, df) -> list[dict]:
        """从 quarterly_financials / financials / quarterly_income_stmt / income_stmt
        中提取历史财报日期，只返回落在 df 时间范围内的日期。
        返回列表元素: {'date': pd.Timestamp(tz=UTC), 'label': str, 'x': pd.Timestamp, 'y': float}
        """
        stock = yf.Ticker(ticker)

        # df.index 统一转为 UTC，用于范围判断和价格查找
        idx_utc = df.index
        if idx_utc.tzinfo is None:
            idx_utc = idx_utc.tz_localize('UTC')
        df_start = idx_utc.min()
        df_end   = idx_utc.max()

        raw: list[dict] = []   # {'date': Timestamp(UTC), 'label': str}

        def _add_from_columns(data, label):
            """从 DataFrame 的列（财报期末日期）提取"""
            try:
                if data is None or (hasattr(data, 'empty') and data.empty):
                    return
                for col in data.columns:
                    try:
                        dt = pd.Timestamp(col)
                        if dt.tzinfo is None:
                            dt = dt.tz_localize('UTC')
                        if df_start <= dt <= df_end:
                            raw.append({'date': dt, 'label': label})
                    except Exception:
                        pass
            except Exception:
                pass

        _add_from_columns(stock.quarterly_financials,  '季度财报')
        _add_from_columns(stock.financials,            '年度财报')
        _add_from_columns(stock.quarterly_income_stmt, '季度利润表')
        _add_from_columns(stock.income_stmt,           '年度利润表')

        # 去重（同一天只保留一条）
        seen: set = set()
        unique: list[dict] = []
        for item in sorted(raw, key=lambda x: x['date']):
            key = item['date'].date()
            if key not in seen:
                seen.add(key)
                # 找 df 中距离最近的交易日价格
                try:
                    loc = idx_utc.searchsorted(item['date'])
                    if loc >= len(df):
                        loc = len(df) - 1
                    item['x'] = df.index[loc]
                    item['y'] = float(df['Close'].iloc[loc])
                    unique.append(item)
                except Exception:
                    pass

        return unique

    def visualize_colab(self):

        SMA_COLORS = [
            '#F4A261', "#170B5C", '#E76F51', '#9B5DE5',
            '#A8DADC', '#F72585', '#4CC9F0', "#89BF14",
        ]
        win_colors = {w: SMA_COLORS[i % len(SMA_COLORS)]
                      for i, w in enumerate(self.sma_windows)}

        res_list = []
        for t in self.tickers:
            res = self._get_computed_df(t)
            if res:
                res_list.append((t, res[0], res[1]))
            else:
                print(f"  \u274c {t} \u6570\u636e\u83b7\u53d6\u5931\u8d25")
        print("\u2705 \u5b8c\u6210\uff0c\u6b63\u5728\u7ed8\u56fe...\n")

        N_SMA    = len(self.sma_windows)
        N_TRACES = 1 + N_SMA + 2 + 1   # close + SMAs + buy/sell + earnings

        fig = go.Figure()
        traces_info = []

        for ticker_idx, (ticker, company_name, df) in enumerate(res_list):
            is_first = (ticker_idx == 0)
            start = len(fig.data)

            current_signal = self.analyze_market(ticker, df.copy())
            signal_emoji = {'BUY': '\U0001f7e2 BUY', 'SELL': '\U0001f534 SELL', 'HOLD': '\U0001f7e1 HOLD'}[current_signal]
            current_price = df['Close'].iloc[-1]

            fig.add_trace(go.Scatter(
                x=df.index, y=df['Close'],
                name='Close Price',
                visible=is_first,
                line=dict(color='#5B8DB8', width=2),
                opacity=0.7,
                legendgroup='close',
                showlegend=is_first,
            ))

            for w in self.sma_windows:
                is_default = w in [self.short_window, self.long_window]
                if is_first:
                    vis = True if is_default else 'legendonly'
                else:
                    vis = False
                fig.add_trace(go.Scatter(
                    x=df.index, y=df[f'SMA_{w}'],
                    name=f'SMA {w}',
                    visible=vis,
                    line=dict(color=win_colors[w], width=1.5),
                    legendgroup=f'sma{w}',
                    showlegend=is_first,
                ))

            buy_mask = df['Position'] == 1.0
            fig.add_trace(go.Scatter(
                x=df[buy_mask].index,
                y=df[f'SMA_{self.short_window}'][buy_mask],
                mode='markers', name='Buy Signal',
                visible=is_first,
                marker=dict(symbol='triangle-up', size=12, color='#2DC653',
                            line=dict(color='white', width=1)),
                legendgroup='buy',
                showlegend=is_first,
            ))

            sell_mask = df['Position'] == -1.0
            fig.add_trace(go.Scatter(
                x=df[sell_mask].index,
                y=df[f'SMA_{self.short_window}'][sell_mask],
                mode='markers', name='Sell Signal',
                visible=is_first,
                marker=dict(symbol='triangle-down', size=12, color='#E63946',
                            line=dict(color='white', width=1)),
                legendgroup='sell',
                showlegend=is_first,
            ))

            # ── 历史财报日期三角标注 ──────────────────────────────────────────
            earn_items = self._get_historical_earnings_dates(ticker, df)
            earn_x     = [d['x'] for d in earn_items]
            earn_y     = [d['y'] for d in earn_items]
            earn_text  = [d['label'] for d in earn_items]
            fig.add_trace(go.Scatter(
                x=earn_x,
                y=earn_y,
                mode='markers',
                name='财报',
                text=earn_text,
                hovertemplate='<b>%{text}</b><br>%{x|%Y-%m-%d}<br>\u20ac%{y:.2f}<extra></extra>',
                visible=is_first,
                marker=dict(symbol='triangle-up', size=11, color='#FFB703',
                            line=dict(color='white', width=1)),
                legendgroup='earnings',
                showlegend=is_first,
            ))

            traces_info.append((start, company_name, ticker, signal_emoji, current_price))

        total_traces = len(fig.data)

        ticker_buttons = []
        for i, (start, company_name, ticker, signal_emoji, price) in enumerate(traces_info):
            vis = []
            showlegend = []
            for j, (s, _, _, _, _) in enumerate(traces_info):
                if j == i:
                    vis.append(True)
                    showlegend.append(True)
                    for w in self.sma_windows:
                        is_default = w in [self.short_window, self.long_window]
                        vis.append(True if is_default else 'legendonly')
                        showlegend.append(True)
                    vis.append(True)   # buy
                    showlegend.append(True)
                    vis.append(True)   # sell
                    showlegend.append(True)
                    vis.append(True)   # earnings
                    showlegend.append(True)
                else:
                    vis.extend([False] * N_TRACES)
                    showlegend.extend([False] * N_TRACES)
            if len(company_name) > 20:
                company_name = company_name[:17] + '...'
            ticker_buttons.append(dict(
                label=company_name,
                method='update',
                args=[
                    {'visible': vis, 'showlegend': showlegend},
                    {'title': f'<b>{company_name}</b>  |  \u20ac{price:.2f}  |  {signal_emoji}'},
                ],
            ))

        first = traces_info[0]
        init_title = f'<b>{first[1]}</b>  |  \u20ac{first[4]:.2f}  |  {first[3]}'

        fig.update_layout(
            title=dict(text=init_title, y=0.97, yanchor='top'),
            height=620,
            template='plotly_white',
            hovermode='x unified',
            xaxis_title='Date',
            yaxis_title='Price (\u20ac)',
            legend=dict(
                orientation='h',
                yanchor='bottom', y=1.06,
                xanchor='right', x=1,
                itemclick='toggle',
                itemdoubleclick='toggleothers',
            ),
            margin=dict(l=50, r=30, t=180, b=50),
            updatemenus=[
                dict(
                    type='dropdown',
                    direction='down',
                    buttons=ticker_buttons,
                    x=0, xanchor='left',
                    y=1.18, yanchor='bottom',
                    showactive=True,
                    bgcolor='white',
                    bordercolor='#aaaaaa',
                    font=dict(size=13),
                ),
            ],
        )

        display(fig)

    # ── Event Agenda ──────────────────────────────────────────────────────────

    @staticmethod
    def _utc_now():
        return datetime.now(tz=timezone.utc)

    def get_events(self, ticker: str, _days: int = 91) -> list[dict]:
        """Return a list of event dicts for the next `_days` days (default 91 ≈ 3 months)."""
        stock  = yf.Ticker(ticker)
        today  = self._utc_now().replace(hour=0, minute=0, second=0, microsecond=0)
        cutoff = today + timedelta(days=_days)
        events: list[dict] = []

        try:
            cal = stock.calendar
            if isinstance(cal, dict):
                for key in ('Earnings Date', 'Earnings High', 'Earnings Low',
                            'Earnings Average', 'Ex-Dividend Date', 'Dividend Date'):
                    val = cal.get(key)
                    if val is None:
                        continue
                    if isinstance(val, list):
                        dates = val
                    elif hasattr(val, '__iter__') and not isinstance(val, str):
                        dates = list(val)
                    else:
                        dates = [val]
                    for d in dates:
                        if d is None:
                            continue
                        try:
                            dt = pd.Timestamp(d)
                            if dt.tzinfo is None:
                                dt = dt.tz_localize('UTC')
                            if today <= dt <= cutoff:
                                label = {
                                    'Earnings Date':    '\U0001f4ca Earnings Release',
                                    'Ex-Dividend Date': '\U0001f4b0 Ex-Dividend',
                                    'Dividend Date':    '\U0001f4b5 Dividend Payment',
                                }.get(key, f'\U0001f4cc {key}')
                                events.append({'date': dt, 'event': label, 'source': 'calendar'})
                        except Exception:
                            pass
            elif isinstance(cal, pd.DataFrame):
                for col in cal.columns:
                    for val in cal[col].dropna():
                        try:
                            dt = pd.Timestamp(val)
                            if dt.tzinfo is None:
                                dt = dt.tz_localize('UTC')
                            if today <= dt <= cutoff:
                                events.append({'date': dt, 'event': f'\U0001f4c5 {col}', 'source': 'calendar'})
                        except Exception:
                            pass
        except Exception:
            pass

        try:
            ed = stock.earnings_dates
            if ed is not None and not ed.empty:
                for idx in ed.index:
                    try:
                        dt = pd.Timestamp(idx)
                        if dt.tzinfo is None:
                            dt = dt.tz_localize('UTC')
                        if today <= dt <= cutoff:
                            row   = ed.loc[idx]
                            eps   = row.get('EPS Estimate', None) if hasattr(row, 'get') else None
                            extra = f'  (EPS est. {eps:.2f})' if eps and not pd.isna(eps) else ''
                            events.append({'date': dt, 'event': f'\U0001f4ca Earnings Release{extra}',
                                           'source': 'earnings_dates'})
                    except Exception:
                        pass
        except Exception:
            pass

        seen   = set()
        unique = []
        for ev in sorted(events, key=lambda x: x['date']):
            key = (ev['date'].date(), ev['event'][:30])
            if key not in seen:
                seen.add(key)
                unique.append(ev)

        return unique

    def _build_events_html(self, ticker: str, company_name: str, events: list[dict], _days: int = 91) -> str:
        today = self._utc_now()
        rows  = ""
        for ev in events:
            dt       = ev['date']
            days_off = (dt.date() - today.date()).days
            if days_off == 0:
                badge = '<span style="background:#2DC653;color:white;border-radius:4px;padding:1px 7px;font-size:11px;">TODAY</span>'
            elif days_off <= 7:
                badge = f'<span style="background:#F4A261;color:white;border-radius:4px;padding:1px 7px;font-size:11px;">in {days_off}d</span>'
            elif days_off <= 30:
                badge = f'<span style="background:#5B8DB8;color:white;border-radius:4px;padding:1px 7px;font-size:11px;">in {days_off}d</span>'
            else:
                badge = f'<span style="color:#888;font-size:12px;">in {days_off}d</span>'

            rows += (
                f'<tr>'
                f'<td style="padding:8px 14px;white-space:nowrap;font-weight:600;color:#333;">{dt.strftime("%d %b %Y")}</td>'
                f'<td style="padding:8px 14px;">{ev["event"]}</td>'
                f'<td style="padding:8px 14px;text-align:center;">{badge}</td>'
                f'</tr>'
            )

        if not rows:
            rows = f'<tr><td colspan="3" style="padding:16px;text-align:center;color:#999;">No scheduled events found for the next {_days} days.</td></tr>'

        return (
            '<div style="font-family:Arial,sans-serif;max-width:700px;margin:12px 0;">'
            '<div style="background:#f7f9fc;border:1px solid #dde3ec;border-radius:8px;padding:10px 18px;margin-bottom:10px;">'
            f'<span style="font-size:16px;font-weight:700;color:#1a1a2e;">{company_name}</span>'
            f'<span style="font-size:13px;color:#888;margin-left:8px;">({ticker})</span>'
            f'<span style="float:right;font-size:12px;color:#aaa;">next {_days} days</span>'
            '</div>'
            '<table style="width:100%;border-collapse:collapse;background:white;border:1px solid #dde3ec;border-radius:8px;overflow:hidden;">'
            '<thead><tr style="background:#1a1a2e;color:white;">'
            '<th style="padding:9px 14px;text-align:left;font-weight:600;">Date</th>'
            '<th style="padding:9px 14px;text-align:left;font-weight:600;">Event</th>'
            '<th style="padding:9px 14px;text-align:center;font-weight:600;">Countdown</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>'
            f'<div style="font-size:11px;color:#bbb;margin-top:6px;text-align:right;">'
            f'Source: Yahoo Finance \u00b7 Updated on {today.strftime("%d %b %Y %H:%M")} UTC</div>'
            '</div>'
        )

    def display_events(self, ticker: str, days: int = 91, output_widget=None) -> None:
        """Fetch and render the event agenda for a ticker."""
        from IPython.display import HTML
        cached       = self._cache.get(ticker)
        company_name = cached[0] if cached else ticker
        events       = self.get_events(ticker, _days=days)
        html         = self._build_events_html(ticker, company_name, events, _days=days)
        if output_widget is not None:
            with output_widget:
                output_widget.clear_output(wait=True)
                display(HTML(html))
        else:
            display(HTML(html))


print('TradingBot class loaded.')

In [ ]:
WATCHLIST = [
    'FR0000133308',   # Orange
    'FR0000120644',   # Danone
    'FR0010908533',   # Edenred SE
    'FR0000121667',   # EssilorLuxottica
    'FR0000120321',   # L'Oréal
    'FR0000120578',   # Sanofi
    'NL0014559478',   # Technip Energies
    'FR0000120271',   # TotalEnergies
    'FR001400AJ45',   # MICHELIN
    'FR0000121709',   # SEB
    'NL0000235190',   # AIRBUS
    'FR0000073272'    # SAFRAN

]

bot = TradingBot(
    tickers=WATCHLIST,
    short_window=10,
    long_window=50,
    sma_windows=[5, 10, 20, 25, 50, 100, 200],
)

print('bot initialized.')

In [ ]:
import ipywidgets as widgets

print('Event Agenda cell loaded.')

In [ ]:
bot.visualize_colab()

In [ ]:
# ── Event Agenda UI ───────────────────────────────────────────────────────────

def _display_name(ticker):
    cached = bot._cache.get(ticker)
    name   = cached[0] if cached else ticker
    return (name[:30] + '...') if len(name) > 33 else name

dd_options = [(f"{_display_name(t)}  ({t})", t) for t in bot.tickers]
dropdown   = widgets.Dropdown(
    options=dd_options,
    value=bot.tickers[0],
    description='Ticker:',
    layout=widgets.Layout(width='380px'),
    style={'description_width': '60px'},
)
out = widgets.Output()

event_days = 365
def _on_change(change):
    if change['name'] == 'value':
        bot.display_events(change['new'], days=event_days, output_widget=out)

dropdown.observe(_on_change, names='value')

display(widgets.VBox([dropdown, out]))
bot.display_events(bot.tickers[0], days=event_days, output_widget=out)

In [ ]:
# bot.run()
# bot.get_company_news(WATCHLIST[0])